# Model, Features, and Outputs (Deep Dive)

This notebook cell documents what the model does, the exact feature engineering used (per session), and the files produced after training and prediction.

## 1) What the model does
- Predicts the final UUT status of a test session (one row in global_metadata) from all steps performed in that session (rows in step_data).
- Targets:
  - Binary: PASS equals 1 and non PASS states (FAIL, DONE, INCOMPLETE) equal 0.
  - Multiclass: predicts one of the original statuses.
- Estimator: Random Forest on a single row per session (per global_id) built from aggregated step behavior, ordering, limits, and context.
- Leakage control for sequence features: FAIL bigram vocabulary is learned only on the training split, then the same indicators are applied to validation and inference.

## 2) Data assumptions
- global_metadata columns: id (global_id), uutStatus, testStarted, testStopped, machineIdentifier, stationName, Testspec, AteSwVersion, etc.
- step_data columns: id, global_id, stepName, stepNumber, stepResult, stepTime, measureValue, limitLow, limitHigh, etc.
- Normalization used:
  - uutStatus uppercased.
  - stepResult uppercased.
  - stepName filled as UNKNOWN if missing.
  - Numeric coercions with errors set to coerce for measureValue and stepTime.
  - Time parsing for testStarted and testStopped using the format YYYY/MM/DD HH:MM:SS.

## 3) Feature engineering (per session)

### A) Step result richness
Captures what failed, how often, and where in the sequence.
- Per stepName by result counts: cnt_FAIL__<step>, cnt_PASS__<step>, cnt_DONE__<step>, cnt_INCOMPLETE__<step>.
- Position buckets by per session stepNumber quantiles (q33 and q66): early, mid, late. Counts by result and bucket such as cnt_early_FAIL, cnt_mid_PASS, cnt_late_INCOMPLETE, etc.
- Fail position stats: first_fail_stepnum, last_fail_stepnum, fail_step_span equals last minus first, fail_clusters equals number of contiguous FAIL segments considering consecutive stepNumber values.
- Any fail per stepName: fail_any__<step> equals 1 if that step failed at least once.
- Global totals: count_FAIL, count_PASS, count_DONE, count_INCOMPLETE, total_steps.

### B) Sequence and order signals
Captures what tends to fail after what.
- Ordered FAIL bigrams are built from each session's FAIL subsequence (e.g., A then B).
- The top K most frequent FAIL bigrams are learned on the training split and saved as top_fail_bigrams.json.
- For each selected pair, create an indicator feature: pairFAIL__<stepA>__then__<stepB> equals 1 if the pair occurs in that session.
- The same bigram vocabulary is used at inference so feature columns align.

### C) Numeric summaries from measurements
- From measureValue coerced to numeric: mean_measureValue, std_measureValue, min_measureValue, max_measureValue, q25_measureValue, q75_measureValue, n_numeric.

### D) Timing summaries and meta time
- From stepTime coerced to numeric (if available): mean_stepTime, std_stepTime.
- From global_metadata timestamps (if available): test_duration_sec equals testStopped minus testStarted, start_hour (0 to 23), start_weekday (0 to 6).

### E) Context and metadata
- One hot encodings of stationName, Testspec, AteSwVersion, machineIdentifier with dummy_na enabled.
- Note: Leakage safe target encoding of context by stepName fail rates is reported in analytics but not used as training features in this version.

### F) Missing data handling
- Numeric NaNs are filled with 0 after aggregation.
- One hot encodings include an explicit NaN category.
- No scaling required for Random Forest.

## 4) Modeling choices
- RandomForestClassifier with default n_estimators set to 300 and class weight balanced for binary targets.
- Stratified train test split.
- Bigrams learned only from the training split to avoid leakage.
- The exact training column order is saved to features_schema.json and also in model.feature_names_in_ when available. Prediction aligns to this order strictly.

## 5) Outputs

### A) Training artifacts in ml_artifacts
- random_forest_<target>.joblib: trained model.
- features_schema.json: n_features, ordered feature_names, and the target type. Used for column alignment at prediction time.
- top_fail_bigrams.json: ordered pairs of FAIL steps selected from the training split that define the sequence features used at train and predict time.
- feature_importance.csv and top_features.csv: impurity importances for all features and top N subset.
- training_dataset_preview.csv: sample of the training matrix with the target for BI or debugging.
- model.feature_names_in_.json: optional debug copy of the model's internal feature ordering.

### B) Prediction results
- predictions.csv and predictions.json with one row per global_id:
  - pred_label equals 0 or 1 for binary, or the original label for multiclass.
  - Probability columns: pred_proba_pass for binary or proba_<CLASS> for multiclass.

### C) Post prediction analytics for failure relations and diagnostics
These use predicted FAIL sessions to reveal what fails, where, and with what.
- predicted_fail_step_patterns.csv and json: stepName and fail_count, the most frequent failed steps among predicted FAIL sessions.
- predicted_fail_step_cooccurrence.csv: unordered pairs of failed steps with co_fail_count, indicating which failures tend to happen in the same session.
- predicted_fail_step_by_bucket.csv: counts of failed steps by position bucket early or mid or late per stepName.
- predicted_fail_bigrams.csv: ordered failure bigrams step_prev to step_next with bigram_count, indicating directional relations.
- predicted_fail_session_failpos.csv: per session first_fail_stepnum, last_fail_stepnum, fail_step_span, fail_clusters.
- predicted_fail_limit_stats_by_step.csv: for steps with numeric limits, rows considered plus mean and median of norm_residual and margin_rel where range equals limitHigh minus limitLow, mid equals half of limitHigh plus limitLow, norm_residual equals measureValue minus mid over range, margin_rel equals the smaller of distance to bounds over range.
- predicted_fail_limit_near_counts_by_step.csv: near5_count and near10_count based on margin_rel thresholds of 0.05 and 0.10, plus numeric_rows processed.
- predicted_fail_rates_by_station_step.csv, predicted_fail_rates_by_testspec_step.csv, predicted_fail_rates_by_atesw_step.csv: for each context and stepName the total rows, fail rows, and fail_rate equals fail over total within predicted FAIL sessions.
- predicted_fail_pacing_session_metrics.csv: per predicted FAIL session from stepTime, mean, median, std, max, total_step_time, time_to_first_fail_sec computed as cumulative stepTime up to the first FAIL, time_to_first_fail_frac relative to test_duration_sec if available else relative to total_step_time, and optionally max_idle_gap_sec if absolute per step timestamps are present.
- predicted_fail_quality_texture.csv: per session counts and ratios for PASS, FAIL, DONE, INCOMPLETE and retry dynamics including total_retries, retry_steps, result_flips across successive attempts of the same stepName, and fail_to_pass_count which indicates steps that failed at least once but ended with PASS.
- predicted_fail_step_retry_summary.csv: per stepName, sessions that ran the step, retry_sessions with more than one attempt, and retry_rate.

## 6) How to read the outputs together
- Start with predicted_fail_step_patterns to identify the largest contributors.
- Use predicted_fail_bigrams for direction and predicted_fail_step_cooccurrence for stable clusters of failures within the same session.
- Check predicted_fail_step_by_bucket and predicted_fail_session_failpos to see where failures arise in the sequence and whether they are early hard failures or late drifts and whether they occur in multiple clusters.
- Inspect the context based tables to spot hot spots for station, testspec, or AteSwVersion by step.
- Use the limit reports to find steps living close to tolerance boundaries.
- Use pacing and quality texture to highlight operational friction such as long times to first fail, idle gaps, retries, and result flips.

## 7) Known limitations
- Tree models do not consume full sequences end to end. This pipeline approximates sequence learning with bigrams and position buckets which usually works well for tabular workflows.
- Context encodings are one hot. Leakage safe target encoding of context by stepName can be promoted to features with proper cross validation if desired.
- Near limit metrics require valid numeric measureValue, limitLow, and limitHigh.

## 8) Reproducibility and alignment
- Prediction enforces the exact training feature order using model.feature_names_in_ when available, else features_schema.json.
- The same top_fail_bigrams.json vocabulary is used at inference to construct matching pairFAIL indicators. If absent, those columns default to zero and the model still runs but without the bigram signal.
